In [22]:
# ==========================
# Step 1: Import Libraries
# ==========================

import pandas as pd
import nltk
import re

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Download NLTK stopwords
nltk.download('stopwords')

# ==========================
# Step 2: Load Dataset
# ==========================

data = pd.read_csv("sentimentdataset.csv")

print("First 5 Rows:")
print(data.head())

print("\nColumns:")
print(data.columns)

# ==========================
# Step 3: Keep Required Columns
# ==========================

data = data[['Text', 'Sentiment']]

# Remove leading/trailing spaces
data['Sentiment'] = data['Sentiment'].str.strip()

# Keep only Positive and Negative reviews
data = data[data['Sentiment'].isin(['Positive', 'Negative'])]

print("\nSentiment Count:")
print(data['Sentiment'].value_counts())

# ==========================
# Step 4: Text Preprocessing
# ==========================

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):

    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Tokenization
    words = text.split()

    # Remove stopwords and apply stemming
    words = [stemmer.stem(word) for word in words if word not in stop_words]

    # Join words
    return " ".join(words)

# Apply preprocessing
data["Clean_Text"] = data["Text"].apply(preprocess)

print("\nCleaned Text:")
print(data[["Text", "Clean_Text"]].head())

# Remove empty rows
data = data.dropna(subset=["Clean_Text"])
data = data[data["Clean_Text"].str.strip() != ""]

# ==========================
# Step 5: TF-IDF Vectorization
# ==========================

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(data["Clean_Text"])

y = data["Sentiment"]

# ==========================
# Step 6: Train-Test Split
# ==========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ==========================
# Step 7: Train Model
# ==========================

model = MultinomialNB()

model.fit(X_train, y_train)

# ==========================
# Step 8: Prediction
# ==========================

prediction = model.predict(X_test)

# ==========================
# Step 9: Evaluation
# ==========================

print("\nAccuracy:")
print(accuracy_score(y_test, prediction))

print("\nClassification Report:")
print(classification_report(y_test, prediction))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, prediction))

# ==========================
# Step 10: Predict New Review
# ==========================

new_review = [
    "This movie is fantastic and I really loved it."
]

# Preprocess
clean_review = [preprocess(new_review[0])]

# Convert into TF-IDF
new_vector = tfidf.transform(clean_review)

# Predict
result = model.predict(new_vector)

print("\nReview:")
print(new_review[0])

print("Predicted Sentiment:", result[0])

# ==========================
# Step 11: Try Another Review
# ==========================

new_review = [
    "The movie was boring and a complete waste of time."
]

clean_review = [preprocess(new_review[0])]

new_vector = tfidf.transform(clean_review)

result = model.predict(new_vector)

print("\nReview:")
print(new_review[0])

print("Predicted Sentiment:", result[0])

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Harshini\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


First 5 Rows:
   Unnamed: 0.1  Unnamed: 0  \
0             0           0   
1             1           1   
2             2           2   
3             3           3   
4             4           4   

                                                Text    Sentiment  \
0   Enjoying a beautiful day at the park!        ...   Positive     
1   Traffic was terrible this morning.           ...   Negative     
2   Just finished an amazing workout! 💪          ...   Positive     
3   Excited about the upcoming weekend getaway!  ...   Positive     
4   Trying out a new recipe for dinner tonight.  ...   Neutral      

             Timestamp            User     Platform  \
0  2023-01-15 12:30:00   User123          Twitter     
1  2023-01-15 08:45:00   CommuterX        Twitter     
2  2023-01-15 15:45:00   FitnessFan      Instagram    
3  2023-01-15 18:20:00   AdventureX       Facebook    
4  2023-01-15 19:55:00   ChefCook        Instagram    

                                     Hashtags  Retwee

C:\Users\Harshini\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
